## EXP-INGEST-004 — Real AI Metadata Enrichment

### Purpose

I will now replace the mock enrichment result with a real AI-generated response.

The Router has already determined the appropriate model tier. CEREBRO will execute the selected capability, capture its provenance and latency, and return structured metadata for human review.

For the PoC, model integration is deliberately isolated so it can be replaced later without changing the CEREBRO workflow.

### 1 — Load artifact

In [12]:
from pathlib import Path
import json
import time

repo_root = Path.cwd().parents[1]

source_path = (
    repo_root
    / "poc/data/raw/text/benchmark_001.txt"
)

assert source_path.exists()

source_text = source_path.read_text(
    encoding="utf-8"
)

print("✓ Artifact loaded")
print("File :", source_path.name)
print("Words:", len(source_text.split()))

✓ Artifact loaded
File : benchmark_001.txt
Words: 41


### 2 — Define the output contract

In [13]:
enrichment_schema = {
    "title": "",
    "artifact_type": "",
    "language": "",
    "description": "",
    "authors": [],
    "people": [],
    "organizations": [],
    "projects": [],
    "topics": [],
    "tags": []
}

enrichment_schema

{'title': '',
 'artifact_type': '',
 'language': '',
 'description': '',
 'authors': [],
 'people': [],
 'organizations': [],
 'projects': [],
 'topics': [],
 'tags': []}

### 3 — Build the prompt

In [14]:
enrichment_prompt = f"""
You are the metadata enrichment capability for CEREBRO,
a Digital Knowledge Twin.

Analyze ONLY the artifact content below.

Return valid JSON with exactly these fields:

{json.dumps(enrichment_schema, indent=2)}

Rules:
- Do not invent information.
- Use empty strings or [] when evidence is insufficient.
- Do not assume the artifact owner is the author.
- Preserve names exactly.
- Topics must be grounded in the artifact.
- Keep the description concise.
- Return JSON only.

ARTIFACT:

{source_text}
""".strip()

print("✓ Enrichment task prepared")

✓ Enrichment task prepared


### 4 — Model adapter

In [15]:
import requests

OLLAMA_URL = "http://localhost:11434"
LOCAL_MODEL = "llama3.2:latest"

def enrich_metadata(prompt):

    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": LOCAL_MODEL,
            "prompt": prompt,
            "stream": False,
            "format": "json",
            "options": {
                "temperature": 0
            }
        },
        timeout=120
    )

    print("HTTP Status:", response.status_code)

    response.raise_for_status()

    result = response.json()

    return {
        "content": result["response"],
        "model": LOCAL_MODEL,
        "provider": "local_ollama"
    }

print(f"✓ Local model configured: {LOCAL_MODEL}")

✓ Local model configured: llama3.2:latest


In [16]:
test_response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2:latest",
        "prompt": 'Return {"status": "ok"}',
        "stream": False,
        "format": "json"
    },
    timeout=60
)

print("Status:", test_response.status_code)
print(test_response.text[:500])

Status: 200
{"model":"llama3.2:latest","created_at":"2026-09-24T04:48:02.380147254Z","response":"{\"status\": \"ok\"}","done":true,"done_reason":"stop","context":[128006,9125,128007,271,38766,1303,33025,2696,25,6790,220,2366,18,271,128009,128006,882,128007,271,5715,5324,2899,794,330,564,9388,128009,128006,78191,128007,271,5018,2899,794,330,564,9388],"total_duration":3336131419,"load_duration":2146749626,"prompt_eval_count":32,"prompt_eval_duration":570296417,"eval_count":7,"eval_duration":526317543}


### 5 — Execute

In [17]:
start_time = time.perf_counter()

model_response = enrich_metadata(
    enrichment_prompt
)

elapsed = time.perf_counter() - start_time

print("✓ AI enrichment complete")
print("Provider :", model_response["provider"])
print("Model    :", model_response["model"])
print("Latency  :", round(elapsed, 3), "seconds")

HTTP Status: 200
✓ AI enrichment complete
Provider : local_ollama
Model    : llama3.2:latest
Latency  : 8.977 seconds


### 6 — Parse structured output

In [18]:
ai_result = json.loads(
    model_response["content"]
)

ai_result

{'title': 'CEREBRO',
 'artifact_type': 'Digital Knowledge Twin',
 'language': '',
 'description': 'Preserves and connects human knowledge, maintaining provenance between fragments and their original source artifacts.',
 'authors': [],
 'people': [''],
 'organizations': [[''], ''],
 'projects': [[''], ''],
 'topics': [['Knowledge Management', 'Digital Twinning']],
 'tags': []}

### 7 — Basic validation

In [19]:
required_fields = set(
    enrichment_schema.keys()
)

returned_fields = set(
    ai_result.keys()
)

missing_fields = (
    required_fields - returned_fields
)

unexpected_fields = (
    returned_fields - required_fields
)

print("Missing fields   :", missing_fields)
print("Unexpected fields:", unexpected_fields)

schema_valid = (
    len(missing_fields) == 0
)

print(
    "\nSchema:",
    "PASS" if schema_valid else "FAIL"
)

Missing fields   : set()
Unexpected fields: set()

Schema: PASS


### 8 — Capture routing + model provenance

In [20]:
ai_prefill = {}

for field, value in ai_result.items():

    ai_prefill[field] = {
        "value": value,

        "source": "ai_suggested",

        "status": "suggested",

        "user_confirmed": False,

        "provenance": {
            "task_id": "TASK-ENRICH-001",

            "router": "cerebro_router",
            "router_version": "0.1",

            "route": "local",

            "provider": model_response[
                "provider"
            ],

            "model": model_response[
                "model"
            ],

            "latency_seconds": round(
                elapsed,
                3
            ),

            "escalated": False
        }
    }

ai_prefill

{'title': {'value': 'CEREBRO',
  'source': 'ai_suggested',
  'status': 'suggested',
  'user_confirmed': False,
  'provenance': {'task_id': 'TASK-ENRICH-001',
   'router': 'cerebro_router',
   'router_version': '0.1',
   'route': 'local',
   'provider': 'local_ollama',
   'model': 'llama3.2:latest',
   'latency_seconds': 8.977,
   'escalated': False}},
 'artifact_type': {'value': 'Digital Knowledge Twin',
  'source': 'ai_suggested',
  'status': 'suggested',
  'user_confirmed': False,
  'provenance': {'task_id': 'TASK-ENRICH-001',
   'router': 'cerebro_router',
   'router_version': '0.1',
   'route': 'local',
   'provider': 'local_ollama',
   'model': 'llama3.2:latest',
   'latency_seconds': 8.977,
   'escalated': False}},
 'language': {'value': '',
  'source': 'ai_suggested',
  'status': 'suggested',
  'user_confirmed': False,
  'provenance': {'task_id': 'TASK-ENRICH-001',
   'router': 'cerebro_router',
   'router_version': '0.1',
   'route': 'local',
   'provider': 'local_ollama',
   '

### 9 — Show the prefilled form

In [21]:
print("CEREBRO — Artifact Review")
print("=" * 50)

for field, metadata in ai_prefill.items():

    value = metadata["value"]

    if value in ["", None, []]:
        value = "[Not determined]"

    print(
        f"{field:15}: {value}"
    )

print("\n" + "-" * 50)
print("Status      : STAGED")
print("AI          : COMPLETE")
print("User Review : PENDING")
print("Submit      : BLOCKED")

CEREBRO — Artifact Review
title          : CEREBRO
artifact_type  : Digital Knowledge Twin
language       : [Not determined]
description    : Preserves and connects human knowledge, maintaining provenance between fragments and their original source artifacts.
authors        : [Not determined]
people         : ['']
organizations  : [[''], '']
projects       : [[''], '']
topics         : [['Knowledge Management', 'Digital Twinning']]
tags           : [Not determined]

--------------------------------------------------
Status      : STAGED
AI          : COMPLETE
User Review : PENDING
Submit      : BLOCKED


### 10 — Final experiment result

In [22]:
assert schema_valid

print("✓ Real model executed")
print("✓ Structured metadata returned")
print("✓ Model provenance captured")
print("✓ Latency measured")
print("✓ AI suggestions remain unconfirmed")

print(
    "\nEXP-INGEST-004: PASS"
)

✓ Real model executed
✓ Structured metadata returned
✓ Model provenance captured
✓ Latency measured
✓ AI suggestions remain unconfirmed

EXP-INGEST-004: PASS


### 11 - Result persistence

In [23]:
from pathlib import Path
import json

repo_root = Path.cwd().parents[1]

output_dir = (
    repo_root
    / "poc/data/processed"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    output_dir
    / "EXP-INGEST-004-ai-prefill.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        ai_prefill,
        f,
        indent=2,
        ensure_ascii=False
    )

print("✓ AI enrichment persisted")
print("Output:", output_path)

✓ AI enrichment persisted
Output: /Users/joeldizon/development/cerebro_dev/cerebro/poc/data/processed/EXP-INGEST-004-ai-prefill.json
